In [ ]:
source("helpers/init.r")
source("helpers/actionable_rules.r")

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load data

In [ ]:
clean  <- fread(paste0(BASE_DATA, "coverage.txt")) %>% rename(ampliseqCoverage = ampliseq500Coverage)

# Pivot panel coverage columns to long format
panel_coverage <- clean %>%
  select(sampleId, evidence = top_level, eventType,
         tso500Coverage, ampliseqCoverage) %>%
  pivot_longer(c(tso500Coverage, ampliseqCoverage),
               names_to = "panel_key", values_to = "coverage") %>%
  mutate(
    panel       = case_when(
      panel_key == "tso500Coverage"   ~ panel_maps[["tso_cov"]],
      panel_key == "ampliseqCoverage" ~ panel_maps[["amp_cov"]]
    ),
    coverage    = as.logical(coverage),
    event_group = eventType
  ) %>%
  select(sampleId, evidence, panel, event_group, coverage)

# WGS covers all events by definition — one row per event, same granularity as panels
wgs_coverage <- clean %>%
  mutate(evidence = top_level, panel = panel_maps[["wgs"]],
         coverage = TRUE, event_group = eventType) %>%
  select(sampleId, evidence, panel, event_group, coverage)

coverage_long <- bind_rows(wgs_coverage, panel_coverage) %>%
  mutate(evidence = case_match(evidence, "Experimental/Other Guidelines" ~ "Experimental", .default = evidence))

# Full sample x evidence x panel grid
ref_ready <-
  bind_rows(
    cohorts %>% filter(!ref) %>% mutate(evidence = "FDA Approved"),
    cohorts %>% filter(!ref) %>% mutate(evidence = "Drug Repurposing"),
    cohorts %>% filter(!ref) %>% mutate(evidence = "Experimental"),
    cohorts %>% filter(!ref) %>% mutate(evidence = "Other")
  )

ref_go <-
  bind_rows(
    ref_ready %>% mutate(panel = panel_maps[["wgs"]]),
    ref_ready %>% mutate(panel = panel_maps[["tso_cov"]]),
    ref_ready %>% mutate(panel = panel_maps[["amp_cov"]])
  )

coverage_data <-
  ref_go %>%
  select(sampleId, cohort, evidence, panel) %>%
  left_join(coverage_long, by = c("sampleId", "evidence", "panel"),
            relationship = "many-to-many") %>%
  mutate(coverage    = ifelse(is.na(coverage),    FALSE,  coverage),
         event_group = ifelse(is.na(event_group), "None", event_group))


# 1 - Event counts

In [ ]:
event_cts <-
  coverage_data %>%
  group_by(sampleId, cohort, panel, evidence, .drop = FALSE) %>%
  summarise(ct = sum(coverage), .groups = "drop") %>%
  mutate(gp  = factor(cut(ct, c(-1, 0, 1, 1000), c("0", "2+", "1")), levels = c("0", "2+", "1")),
         gp2 = factor(cut(ct, c(-1, 0, 1000),     c("0", "1+")),      levels = c("0", "1+")),
         panel    = factor(panel,    levels = rev(unname(unlist(panel_maps)))),
         evidence = factor(evidence, levels = names(level_map))) %>%
  group_by(cohort, panel) %>%
  mutate(tot = n_distinct(sampleId)) %>%
  ungroup()

# 2 - Average events plot

In [ ]:
stuff <- list(
  geom_bar(stat = "identity", color = "black", alpha = .7),
  base_theme,
  theme(legend.position = "bottom",
        axis.title.y = element_blank(),
        plot.title = element_text(hjust = .5, size = 14),
        plot.margin = margin(l = .5, r = 0, b = .5, t = .5, "cm"),
        panel.grid.major = element_blank(), panel.grid.minor = element_blank()),
  guides(fill = guide_legend(reverse = TRUE))
)

avg_ready <-
  event_cts %>%
  group_by(cohort, panel, evidence, tot) %>%
  summarise(avg_events = mean(ct), .groups = "drop") %>%
  mutate(evidence = factor(evidence, levels = rev(names(level_map))),
         label    = ifelse(avg_events > .2, as.character(signif(avg_events, 2)), ""),
         panel    = factor(panel, levels = rev(unname(unlist(panel_maps)))))

plot_average <- function(cohort_name) {
  base <- avg_ready %>% filter(cohort == cohort_name)
  totals <- base %>% group_by(panel) %>% summarise(avg_events = sum(avg_events))
  ggplot(base, aes(y = panel, x = avg_events, fill = evidence)) +
    stuff +
    scale_fill_manual(values = unlist(actionable_fill_map)) +
    geom_text(aes(label = label), size = 4, position = position_stack(vjust = 0.5)) +
    geom_text(data = totals, inherit.aes = FALSE,
              aes(label = round(avg_events, 1), x = avg_events, y = panel), hjust = -.3, size = 5) +
    labs(fill = NULL, x = "Average actionable events per patient") +
    ggtitle("Average potentially actionable events by test strategy") +
    scale_x_continuous(expand = c(0, 0),
                       limits = c(0, max(totals$avg_events, na.rm = TRUE) * 1.1)) +
    theme(legend.position = "none")
}

avg_plt <- plot_average(plt_cohort)

# 3 - Best treatment option plot

In [ ]:
bto_ready <-
  event_cts %>%
  group_by(cohort, sampleId, panel) %>%
  mutate(event     = sum(ct) > 0,
         evidence  = factor(ifelse(!event, "None", as.character(evidence)), levels = names(level_map)),
         ct        = ifelse(evidence == "None", 1, ct)) %>%
  filter(ct > 0) %>%
  group_by(cohort, sampleId, panel) %>%
  slice_min(evidence, n = 1) %>%
  group_by(cohort, panel, evidence) %>%
  summarise(ct = n(), .groups = "drop") %>%
  filter(evidence != "None") %>%
  mutate(evidence = factor(evidence, levels = rev(names(level_map)))) %>%
  group_by(cohort, panel) %>%
  mutate(pct   = ct / sum(ct),
         label = ifelse(pct > .03, as.character(ct), ""),
         panel = factor(panel, levels = rev(unname(unlist(panel_maps)))))

plot_best_treatment <- function(cohort_name) {
  base   <- bto_ready %>% filter(cohort == cohort_name)
  totals <- base %>% group_by(panel) %>% summarise(ct = sum(ct))
  ggplot(base, aes(y = panel, x = ct, fill = evidence)) +
    stuff +
    scale_fill_manual(values = unlist(actionable_fill_map)) +
    geom_text(aes(label = label), size = 3.5, position = position_stack(vjust = 0.5)) +
    geom_text(data = totals, inherit.aes = FALSE,
              aes(label = ct, x = ct, y = panel), hjust = -.3, size = 5) +
    labs(fill = NULL, x = "Number of patients") +
    ggtitle("Highest evidence level treatment option by test strategy") +
    scale_x_continuous(expand = c(0, 0),
                       limits = c(0, max(totals$ct, na.rm = TRUE) * 1.15))
}

bto_plt <- plot_best_treatment(plt_cohort)

# 4 - Missed events plots

In [ ]:
missed_ready <-
  coverage_data %>%
  filter(cohort == plt_cohort) %>%
  group_by(cohort, panel, evidence, event_group) %>%
  summarise(total_events  = n(),
            missed_events = sum(!coverage),
            .groups = "drop") %>%
  group_by(cohort, panel, evidence) %>%
  mutate(rk       = row_number(desc(missed_events)),
         evidence = factor(evidence, levels = names(level_map))) %>%
  drop_na(event_group) %>%
  filter(event_group != "None")

plot_missed <- function(panel_name, title) {
  dat <-
    missed_ready %>%
    filter(panel == panel_name, tolower(evidence) != "none", rk < 6,
           missed_events > 0) %>%
    mutate(label       = as.character(missed_events),
           event_group = factor(paste(event_group, evidence, sep = "__"),
                           levels = rev(unique(paste(event_group[order(rk)], evidence[order(rk)], sep = "__")))))
  if (nrow(dat) == 0) return(ggplot() + theme_void() +
    labs(title = title) + theme(plot.title = element_text(hjust = .5, size = 14)))
  ggplot(dat, aes(y = event_group, x = missed_events, fill = evidence)) +
    stuff +
    theme(legend.position = "none") +
    scale_fill_manual(values = unlist(actionable_fill_map)) +
    facet_wrap(~evidence, scales = "free_y", ncol = 2) +
    geom_text(aes(label = label), hjust = -.2) +
    scale_y_discrete(labels = function(x) sub("__.*", "", x)) +
    scale_x_continuous(expand = c(0, 0),
                       limits = c(0, max(dat$missed_events, na.rm = TRUE) * 1.3)) +
    ggtitle(title) +
    xlab("Number of missed events")
}

coverage_tso <- plot_missed(panel_maps[["tso_cov"]], "Top missed events: Comprehensive panel vs WGS")
coverage_amp <- plot_missed(panel_maps[["amp_cov"]], "Top missed events: Targeted panel vs WGS")

# 5 - Save figures

In [ ]:
figures <- list(avg = avg_plt, bto = bto_plt, tso = coverage_tso, amp = coverage_amp)
saveRDS(figures, paste0(READY_FIG, "coverage.rds"))